# C-MAPSS Exploratory Data Analysis

Phase 1 deliverable. Profiles all four NASA C-MAPSS subsets (FD001–FD004) to motivate
the preprocessing choices made in `src/cmapss_rul/data/`:

1. **RUL distribution** — verifies the piecewise-linear clip at 125 cycles.
2. **Sensor variance per subset** — confirms which sensors are constant and dropped.
3. **Operating-condition clusters** — confirms the 6-cluster assumption for FD002/FD004.
4. **Trajectory length per engine** — informs the 30-timestep window choice.

All cells expect raw C-MAPSS files in `data/raw/`. See `data/README.md` for the download link.
If raw data is absent the cells below will raise `FileNotFoundError` — that is the intended signal,
not a notebook bug.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from cmapss_rul.config import CMAPSS_SUBSETS, DEFAULT_RUL_CLIP
from cmapss_rul.data.loader import SENSOR_COLUMNS, SETTING_COLUMNS, load_subset
from cmapss_rul.data.preprocessing import compute_rul, find_constant_columns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Load all four training subsets

In [ ]:
subsets: dict[str, pd.DataFrame] = {s: load_subset(s, split="train") for s in CMAPSS_SUBSETS}
summary = pd.DataFrame(
    {
        "n_rows": {s: len(df) for s, df in subsets.items()},
        "n_units": {s: df['unit'].nunique() for s, df in subsets.items()},
        "mean_cycles_per_unit": {
            s: round(df.groupby('unit')['cycle'].max().mean(), 1)
            for s, df in subsets.items()
        },
    }
)
summary

## 2. RUL distribution under the piecewise-linear clip

Heimes (2008) and most subsequent papers cap RUL at 125 cycles because early-life
RUL targets are not learnable from sensor signals (no degradation yet). The histograms
below should show a flat-topped distribution at 125 with mass decaying toward 0.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
for ax, (name, df) in zip(axes.flat, subsets.items()):
    rul = compute_rul(df, clip=DEFAULT_RUL_CLIP)
    ax.hist(rul, bins=40, color="#2E5A88", alpha=0.8)
    ax.set_title(f"{name} — RUL (clipped at {DEFAULT_RUL_CLIP})")
    ax.set_xlabel("RUL (cycles)")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Constant sensors per subset

Sensors with zero variance contribute nothing to RUL prediction and break
min-max normalization (zero range). The pipeline drops them automatically;
this cell makes the drop list visible.

In [ ]:
constant_table = pd.DataFrame(
    {s: {col: col in find_constant_columns(df, list(SENSOR_COLUMNS)) for col in SENSOR_COLUMNS}
     for s, df in subsets.items()}
)
constant_table.loc[constant_table.any(axis=1)]

## 4. Operating-condition clusters (FD002, FD004)

The 6-cluster assumption baked into `OperatingConditionNormalizer` comes from
Saxena et al. (2008). The scatter of `setting_1` vs `setting_2` should show
clear discrete clusters for FD002/FD004 and a single tight cluster for FD001/FD003.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (name, df) in zip(axes.flat, subsets.items()):
    ax.scatter(df['setting_1'], df['setting_2'], s=2, alpha=0.3, color="#2E5A88")
    ax.set_title(f"{name} — operating settings")
    ax.set_xlabel("setting_1")
    ax.set_ylabel("setting_2")
plt.tight_layout()
plt.show()

## 5. Engine trajectory lengths

Window size = 30 was chosen to fit the shortest expected trajectory. The
histogram below verifies most units have ≥ 30 cycles; the few shorter ones
are padded by `make_windows` (see `preprocessing.py`).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for name, df in subsets.items():
    lengths = df.groupby('unit')['cycle'].max()
    ax.hist(lengths, bins=30, alpha=0.5, label=name)
ax.axvline(30, color='red', linestyle='--', label='window_size=30')
ax.set_xlabel('Cycles per engine')
ax.set_ylabel('Count')
ax.set_title('Trajectory length distribution')
ax.legend()
plt.show()

## Conclusions

- FD001 / FD003: single operating condition → `GlobalNormalizer` is sufficient.
- FD002 / FD004: ~6 operating-condition clusters visible → `OperatingConditionNormalizer` required.
- Constant sensors (1, 5, 6, 10, 16, 18, 19 in FD001/FD003) are dropped automatically.
- A 30-timestep window covers all trajectories with minimal padding.
- RUL clip at 125 produces a clean, learnable target distribution.

These observations are not hand-tuned in `src/cmapss_rul/data/` — they fall out of the
implementation. The pipeline picks the right normalizer per subset, drops constant sensors
from training-set variance, and pads short trajectories. Reproducibility is captured in
`data/processed/manifest.json` (git SHA + raw-data SHA-256 + per-subset stats).